# 3-2 Assignment: Identifying CIFAR-10 Images
---
<div class="alert alert-block alert-success" style="color:black;">
<b>To Begin:</b> Run all code blocks and observe the output. Once you have reviewed the sample output. Use the <b>LastName_FirstName_Assignment2.ipynb</b> file to complete your assignment.
</div>

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> For compatability purposes, libraries have been updated from those used in the required readings to match to current versions; hence some of the package invocations may differ slightly from the book. The affected lines of code have comments added to the right as applicable, with the old code commented out above for reference.
</div>

<div class="alert alert-block alert-danger" style="color:black;">
<b>GPU/CUDA/Memory Warnings/Errors:</b> You may receive some errors referencing that GPUs will not be used, CUDA could not be found, or free system memory allocation errors. These and a few others, are standard errors that can be ignored here as they are environment based.<br><br>
<b>Example messages:</b>
    <ul>
        <li>Could not find cuda drivers on your machine, GPU will not be used.</li>
        <li>Please check linkage and avoid linking the same target more than once.</li>
        <li>E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)</li>
        <li>Allocation of ######## exceeds 10% of free system memory</li>
    </ul>
</div>

---

### Installing Required Packages
This is to install necessary components to run the assignment

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from keras.datasets import cifar10
from keras.utils import to_categorical # Syntax change due to version bump
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Flatten # Syntax change due to version bump
from keras.layers import Conv2D, MaxPooling2D
from keras.optimizers import SGD, Adam, RMSprop
import matplotlib.pyplot as plt
import tensorflow as tf
import os # For saving model purposes

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b> # Logs that appear below are informational!
</div>


In [3]:
# CIFAR_10 is a set of 60K images 32 x 32 pixels on 3 channels
IMG_CHANNELS = 3
IMG_ROWS = 32
IMG_COLS = 32

# Constant 
BATCH_SIZE = 128
NB_EPOCH = 20
NB_CLASSES = 10
VERBOSE = 1
VALIDATION_SPLIT = 0.2
OPTIM = RMSprop()

I0000 00:00:1790366584.828308     414 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790366586.236336     414 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790366586.239814     414 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1790366586.244039     414 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    If an UNKNOWN ERROR LOG DISPLAYS, it's just tensorflow trying to connect to GPU. <br>
    It'll simply redirect and use the CPU!
</div>


In [4]:
# Load the datasets
(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print('X_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

X_train shape: (50000, 32, 32, 3)
50000 train samples
10000 test samples


<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    One-hot Encoding & Normalization of images<br>
    In the book, the older version is:<br>
    Y_train = np_utils.to_categorical(y_train, NB_CLASSES)
</div>


In [5]:
Y_train = to_categorical(y_train, NB_CLASSES)
Y_test = to_categorical(y_test, NB_CLASSES)
                        
# float and normalization
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
X_train /= 255
X_test /= 255

In [6]:
# Network
model = Sequential()
model.add(Conv2D(32, (3, 3), padding="same",
                input_shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS)))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(NB_CLASSES))
model.add(Activation('softmax'))
model.summary()
# Below red is simply a userwarning. 

/home/codio/.pyenv/versions/3.11.9/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     4,194,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,200,842 (16.02 MB)

 Trainable params: 4,200,842 (16.02 MB)

 Non-trainable params: 0 (0.00 B)

<div class="alert alert-block alert-warning" style="color:black;">
    <b>Note:</b><br>
    <b><i>Training...</i></b> <br><br>NOTE: THIS MAY TAKE SOME TIME. Go grab a cup of coffee!
</div>


In [7]:
model.compile(loss="categorical_crossentropy", optimizer=OPTIM, metrics=["accuracy"])

history = model.fit(X_train, Y_train, batch_size=BATCH_SIZE)
score = model.evaluate(X_test, Y_test, batch_size=BATCH_SIZE, verbose=VERBOSE)
print('Test Score:', score[0])
print('Test Accuracy:', score[1])

I0000 00:00:1790366592.724143     478 service.cc:146] XLA service 0x73b280007950 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790366592.724172     478 service.cc:154]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5


 42/391 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1573 - loss: 2.5444

I0000 00:00:1790366595.126444     478 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


391/391 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.3074 - loss: 1.9606
79/79 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.5275 - loss: 1.3523
Test Score: 1.3538095951080322
Test Accuracy: 0.5282999873161316


In [8]:
# Save the model
os.makedirs('output', exist_ok=True)
model_json = model.to_json()
open('./output/cifar10_architecture.json', 'w').write(model_json)

# And the weights learned by our deep network on the training set
model.save_weights('./output/cifar10.weights.h5', overwrite=True) # NOTE, it is now cifar10.weights, not cifar10_weights

In [9]:
# Improving the CIFAR-10 performance with deeper network
model = Sequential()
model.add(Conv2D(32, (3,3), padding='same', input_shape=(IMG_ROWS, IMG_COLS, IMG_CHANNELS)))
model.add(Activation('relu'))
model.add(Conv2D(32, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))
model.add(Conv2D(64, (3, 3), padding='same'))
model.add(Activation('relu'))
model.add(Conv2D(64, 3, 3))
model.add(Activation('relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.25))
model.add(Flatten())
model.add(Dense(512))
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(NB_CLASSES))
model.add(Activation('softmax'))

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 5, 5, 64)       │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_6 (Activation)       │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 2, 2, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_7 (Activation)       │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         5,130 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_8 (Activation)       │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 202,282 (790.16 KB)

 Trainable params: 202,282 (790.16 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# Improving the CIFAR-10 performance with data augmentation
from tensorflow.keras.preprocessing.image import ImageDataGenerator # Note, we are using Tensorflow's Keras package!
from keras.datasets import cifar10
import numpy as np
NUM_TO_AUGMENT = 5

In [11]:
# load dataset
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Augmenting
print("Augmenting training set images...")
datagen = ImageDataGenerator(rotation_range=40,
                            width_shift_range=0.2,
                            height_shift_range=0.2,
                            zoom_range=0.2,
                            horizontal_flip=True,
                            fill_mode='nearest')

Augmenting training set images...


<div class="alert alert-block alert-warning" style="color:black;">
    The Below will run to create images from the above ImageDataGenerator(). <br>
    <b>Note:</b> <b><i>If you run it, it will take awhile!</b></i>
</div>


In [12]:
xtas, ytas = [], []

for i in range(X_train.shape[0]):
    num_aug = 0
    x = X_train[i] #(3, 32, 32)
    x = x.reshape((1,) + x.shape) # (1 , 3, 32, 32)
    for x_aug in datagen.flow(x, batch_size=1, save_to_dir='preview', save_prefix='cifar', save_format='jpeg'):
        if num_aug >= NUM_TO_AUGMENT:
            break
        xtas.append(x_aug[0])
        num_aug += 1

In [13]:
optimizer = RMSprop() # Recreating the optimizer

#for the datagen
datagen.fit(X_train)

model.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

# train changed from model.fit_generator()
history = model.fit(
    datagen.flow(X_train, Y_train, batch_size=BATCH_SIZE),
    steps_per_epoch=X_train.shape[0] // BATCH_SIZE,
    epochs=NB_EPOCH,
    validation_data=tf.data.Dataset.from_tensor_slices((X_test, Y_test)).batch(BATCH_SIZE),
    verbose=VERBOSE
)

# score = model.evaluate(X_test, Y_test, batch_size=BATCH_SIZE, verbose=VERBOSE)

print("Test Score:", score[0])
print("Test Accuracy:", score[1])

Epoch 1/20


/home/codio/.pyenv/versions/3.11.9/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


390/390 ━━━━━━━━━━━━━━━━━━━━ 34s 75ms/step - accuracy: 0.1150 - loss: 3.1108 - val_accuracy: 0.2291 - val_loss: 2.0752
Epoch 2/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step - accuracy: 0.2188 - loss: 2.1435 - val_accuracy: 0.2405 - val_loss: 1.9902
Epoch 3/20


/home/codio/.pyenv/versions/3.11.9/lib/python3.11/contextlib.py:158: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


390/390 ━━━━━━━━━━━━━━━━━━━━ 26s 66ms/step - accuracy: 0.2333 - loss: 2.0655 - val_accuracy: 0.3455 - val_loss: 1.7876
Epoch 4/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 440us/step - accuracy: 0.3203 - loss: 1.8172 - val_accuracy: 0.3527 - val_loss: 1.7650
Epoch 5/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 26s 66ms/step - accuracy: 0.2941 - loss: 1.9132 - val_accuracy: 0.3699 - val_loss: 1.7411
Epoch 6/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 452us/step - accuracy: 0.2422 - loss: 1.9047 - val_accuracy: 0.3842 - val_loss: 1.7094
Epoch 7/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 26s 66ms/step - accuracy: 0.3202 - loss: 1.8382 - val_accuracy: 0.3924 - val_loss: 1.6661
Epoch 8/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 445us/step - accuracy: 0.3359 - loss: 1.7688 - val_accuracy: 0.4366 - val_loss: 1.5880
Epoch 9/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 26s 65ms/step - accuracy: 0.3475 - loss: 1.7885 - val_accuracy: 0.4091 - val_loss: 1.5789
Epoch 10/20
390/390 ━━━━━━━━━━━━━━━━━━━━ 0s 443us/step - accuracy: 0.3906 - loss: 1.6749 - val_accura

# MARKDOWN WORK HERE

- Analyze the ethical and privacy implications of the algorithm you just created here!

The CNN trained on the dataset achieved a test accuracy of 52.83% after 20 epochs. The validation accuracy improved during the training, increasing from 22.91% in the first epoch to 50.21% in epoch 19 before dropping to 48.15% in epoch 20. The validation loss also decreased from 2.0752 in the first epoch to 1.3697 in epoch 19 before increasing slightly to 1.4196 in the final epoch. The model used data augmentation to train on modified versions of the images. The results show that the model was able to learn patterns from the CIFAR-10 images, but the 52.83% test accuracy also shows that the model was still making incorrect classifications. There was also a warning during training that the input ran out of data, which is something to consider when evaluating the training results. 

There are also privacy and ethical concerns when neural networks are used for image recognition involving people. A person's face can be considered personally identifiable information (PII), so images need to be handled carefully (Imperva, n.d.). If an organization collects images without a person's knowledge or consent and connects those images to names or other identifying information, it can create significant privacy concerns. Hill (2020) discusses these concerns in relation to Clearview AI and the collection of facial images for law enforcement purposes. Accuracy is another concern because an incorrect identification could have serious consequences. Research has found differences in facial-recognition accuracy across demographic groups, including lower accuracy for darker-skinned women compared with lighter-skinned groups (Najibi, 2020). Automated license plate readers provide another example of how image-recognition technology can collect information about people, including license plate numbers, locations, dates, and times. The Electronic Frontier Foundation has raised concerns about how this information can be used to track people's movements and has documented cases involving errors in license plate recognition (Electronic Frontier Foundation, 2020). These examples show why developers should consider accuracy, bias, privacy, consent, and how information will be stored and used before applying image-recognition technology to real-world situations.

References

Electronic Frontier Foundation. (2020, May 15). Automated license plate readers (ALPRs). https://www.eff.org/pages/automated-license-plate-readers-alpr

Hill, K. (2020, January 21). The secretive company that might end privacy as we know it. International New York Times.

Imperva. (n.d.). What is personally identifiable information (PII)? https://www.imperva.com/learn/data-security/personally-identifiable-information-pii/

Najibi, A. (2020). Racial discrimination in face recognition technology. Science in the News. https://sitn.hms.harvard.edu/flash/2020/racial-discrimination-in-face-recognition-technology/

Sarabdeen, J. (2022, March). Protection of the rights of individuals when using facial recognition technology. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC8927940/

